# Solution Exercise 2.1

In [1]:
import numpy as np
import pandas as pd

In [2]:
# a) Load and quickly inspect the dataset
path_data = "/home/vincent/repos/isc-data-processing-visualization/data/iris_to_clean.csv"
df = pd.read_csv(path_data)
df.head()


,measurement.number,sepal.length,sepal.width,petal.length,petal.width,variety
0,1,5.1,3.5,1.4,0.2,Setosa
1,2,4.9,3.0,1.4,0.2,Setosa
2,3,4.7,3.2,1.3,0.2,Setosa
3,4,4.6,3.1,1.5,0.2,Setosa
4,5,5.0,3.6,1.4,0.2,Setosa


In [3]:
df.describe()

,measurement.number,sepal.length,sepal.width,petal.length,petal.width
count,154.000000,154.000000,153.000000,152.000000,154.000000
mean,74.902597,6.198052,3.211111,4.080921,1.316234
std,43.030886,4.282499,1.984000,4.204818,1.619064
min,1.000000,4.300000,2.000000,1.000000,0.100000
25%,39.250000,5.100000,2.800000,1.600000,0.300000
50%,73.500000,5.800000,3.000000,4.400000,1.300000
75%,111.750000,6.400000,3.300000,5.100000,1.800000
max,150.000000,58.000000,27.000000,51.000000,19.000000


In [ ]:
# b) Check if the dataset contains duplicates. Remove them if any.
# Check in the measurement.number column
col = 'measurement.number'
print(f'Number of duplicates in {col}: {df.duplicated(col).sum()}')
# Or check in the entire df 
print(f'Number of duplicates in entire df: {df.duplicated().sum()}')

Number of duplicates in measurement.number: 4
Number of duplicates in entire df: 4


In [ ]:
# Drop duplicates
df.drop_duplicates(inplace=True) # inplace instead of df = df.drop_duplicate
# Or drop duplicates from the measurement.number colum 
# df.drop_duplicates('measurement.number',inplace=True)
print(f'Number of duplicates in {col}: {df.duplicated(col).sum()}')

Number of duplicates in measurement.number: 0


In [6]:
# c) Check if the dataset contains missing values. Sort them with most missing on top and compute percentages of missingness. 
# Print rows with missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 153
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   measurement.number  150 non-null    int64  
 1   sepal.length        150 non-null    float64
 2   sepal.width         149 non-null    float64
 3   petal.length        148 non-null    float64
 4   petal.width         150 non-null    float64
 5   variety             150 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 8.2+ KB


In [7]:
# Check for missing values.
numerical_columns = ["sepal.length", "sepal.width", "petal.length", "petal.width"]
missing_values = df[numerical_columns].isnull().sum()

# Display the results
print("Missing values in each column:")
print(missing_values)
# sorting missing values and computing the percentage can be useful for larger datasets
print("Sorting percentages missing values :")
print(missing_values.sort_values(ascending=False)/len(df))

# Get rows with missing values in the specified columns
rows_with_missing_values = df[df[numerical_columns].isnull().any(axis=1)]
# Display the rows with missing values
print("Rows with missing values:")
print(rows_with_missing_values)

# Only a few missing values, we will impute values (we could also delete the few columns with dropna())

Missing values in each column:
sepal.length    0
sepal.width     1
petal.length    2
petal.width     0
dtype: int64
Sorting percentages missing values :
petal.length    0.013333
sepal.width     0.006667
sepal.length    0.000000
petal.width     0.000000
dtype: float64
Rows with missing values:
     measurement.number  sepal.length  sepal.width  petal.length  petal.width  \
26                   27           5.0          NaN           1.6          0.4   
36                   37           5.5          3.5           NaN          0.2   
134                 131           7.4          2.8           NaN          1.9   

       variety  
26      Setosa  
36      Setosa  
134  Virginica  


In [8]:
# Impute values by calculating the mean (approximately normally distributed data)
for col in numerical_columns:
    mean_col = round(df[col].mean(),1)
    df[col].fillna(mean_col, inplace=True)

# # Or do the same with scikit-learn
# from sklearn.impute import SimpleImputer
# imputer = SimpleImputer(strategy="mean")
# imputer.fit_transform(df[numerical_columns])
# df[numerical_columns] = imputer.transform(df[numerical_columns])

# Display the rows with missing values again
rows_with_missing_values = df[df[numerical_columns].isnull().any(axis=1)]
print("Rows with missing values:")
print(rows_with_missing_values)

Rows with missing values:
Empty DataFrame
Columns: [measurement.number, sepal.length, sepal.width, petal.length, petal.width, variety]
Index: []


In [9]:
# d) Check if the dataset contains outliers.
for column in ["sepal.length", "sepal.width", "petal.length", "petal.width"]:
    # Calculate the IQR (Interquartile Range)
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define the lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify outliers using the bounds
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    
    # Display the results
    print(f"Outliers in {column}:")
    print(outliers)

Outliers in sepal.length:
     measurement.number  sepal.length  sepal.width  petal.length  petal.width  \
146                 143          58.0         27.0          51.0         19.0   

       variety  
146  Virginica  
Outliers in sepal.width:
     measurement.number  sepal.length  sepal.width  petal.length  petal.width  \
15                   16           5.7          4.4           1.5          0.4   
32                   33           5.2          4.1           1.5          0.1   
33                   34           5.5          4.2           1.4          0.2   
64                   61           5.0          2.0           3.5          1.0   
146                 143          58.0         27.0          51.0         19.0   

        variety  
15       Setosa  
32       Setosa  
33       Setosa  
64   Versicolor  
146   Virginica  
Outliers in petal.length:
     measurement.number  sepal.length  sepal.width  petal.length  petal.width  \
146                 143          58.0         27.0

In [10]:
# Some seem true outliers (represent natural variations in the sample). 
# One should be removed (146)
df.drop(146,axis=0,inplace=True)

In [11]:
# Save the cleaned dataset, check in excel or else that it seems fine
df.to_csv('../data/iris_cleaned.csv', index=False)